# 02 — Dérive génétique et différenciation spatiale (Fst)

**Objectif** : démontrer deux phénomènes fondamentaux de la génétique des populations dans EcoSim :

1. **Dérive génétique** — sur N réplicats partageant les mêmes paramètres mais des seeds différents, l'hétérozygotie attendue He(t) calculée sur les gènes neutres diverge entre réplicats et tend à diminuer dans le temps (perte d'allèles par fluctuation aléatoire).
2. **Différenciation spatiale (Fst)** — sur un seul réplicat, on calcule Fst de Wright entre les quadrants géographiques de la carte. À mesure que les sous-populations s'isolent (faible dispersion), Fst croît : les fréquences alléliques divergent.

Les métriques sont fournies par `ecosim.research.analysis.genetics_metrics` :
- `heterozygosity_expected(genomes)` — He = 1 - Σ pᵢ² moyenné sur tous les loci neutres.
- `fst(pop_a, pop_b)` — Fst entre deux groupes (variance des fréquences alléliques).

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import ecosim as _ecosim_pkg
from ecosim.engine.api import SimConfig, Simulation
from ecosim.research.analysis.genetics_metrics import fst, heterozygosity_expected
from ecosim.version import __version__ as ECOSIM_VERSION

SPECIES_DIR = Path(_ecosim_pkg.__file__).parent / "data" / "species"

def load_species_params(name: str) -> dict:
    spec = json.loads((SPECIES_DIR / f"{name.lower()}.json").read_text(encoding="utf-8"))
    params = spec.get("params", spec)
    if "color" in params:
        params["color"] = tuple(params["color"])
    return params

CONFIG = {
    "ecosim_version":  ECOSIM_VERSION,
    "seeds":           [1, 2, 3, 4, 5],         # réplicats pour la partie drift
    "grid_size":       100,
    "terrain_preset":  "temperate",
    "ticks_total":     4000,
    "checkpoint_every": 250,
    "fst_seed":        42,                       # seed dédié à la partie Fst
    "species": {
        "Herbe": {"count": 400},
        "Lapin": {"count": 60},
    },
}
print(f"ecosim version : {ECOSIM_VERSION}")
print(f"Réplicats : {len(CONFIG['seeds'])} seeds, {CONFIG['ticks_total']} ticks, grille {CONFIG['grid_size']}²")

## 1. Dérive génétique — He(t) sur N réplicats

Pour chaque seed, on lance une simulation et on enregistre He calculé sur les **gènes neutres** des lapins vivants à intervalles réguliers. Les gènes neutres n'affectent pas le phénotype : leur évolution reflète uniquement la dérive (mutation + échantillonnage aléatoire à la reproduction).

À population finie, He doit en moyenne diminuer (perte d'allèles), et la variance entre réplicats doit augmenter — les deux signatures de la dérive.

In [ ]:
def he_of(sim, species_name: str) -> float:
    genomes = [ind.genome for ind in sim.individuals if ind.species.name == species_name]
    return heterozygosity_expected(genomes) if genomes else 0.0

checkpoints = list(range(0, CONFIG["ticks_total"] + 1, CONFIG["checkpoint_every"]))
he_series: dict[int, list[float]] = {seed: [] for seed in CONFIG["seeds"]}
pop_series: dict[int, list[int]]  = {seed: [] for seed in CONFIG["seeds"]}

for seed in CONFIG["seeds"]:
    cfg = SimConfig(seed=seed, grid_size=CONFIG["grid_size"],
                    terrain_preset=CONFIG["terrain_preset"], out_path=None)
    sim = Simulation(cfg)
    for name, opts in CONFIG["species"].items():
        sim.add_species(load_species_params(name), count=opts["count"])
    # Premier point = état initial.
    he_series[seed].append(he_of(sim, "Lapin"))
    pop_series[seed].append(sim.populations.get("Lapin", 0))
    for k in range(1, len(checkpoints)):
        sim.run(checkpoints[k] - checkpoints[k - 1])
        he_series[seed].append(he_of(sim, "Lapin"))
        pop_series[seed].append(sim.populations.get("Lapin", 0))
    print(f"seed={seed} terminé — He final = {he_series[seed][-1]:.4f}, pop = {pop_series[seed][-1]}")

In [ ]:
ticks = np.asarray(checkpoints)
he_matrix  = np.array([he_series[s]  for s in CONFIG["seeds"]])  # (n_replicats, n_checkpoints)
pop_matrix = np.array([pop_series[s] for s in CONFIG["seeds"]])

fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
for i, seed in enumerate(CONFIG["seeds"]):
    axes[0].plot(ticks, he_matrix[i], alpha=0.55, label=f"seed={seed}")
mean = he_matrix.mean(axis=0)
std  = he_matrix.std(axis=0)
axes[0].plot(ticks, mean, color="black", linewidth=2.0, label="moyenne")
axes[0].fill_between(ticks, mean - std, mean + std, color="gray", alpha=0.25, label="±σ")
axes[0].set_ylabel("He (gènes neutres)")
axes[0].set_title(f"EcoSim {ECOSIM_VERSION} — Dérive génétique sur {len(CONFIG['seeds'])} réplicats (Lapin)")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="lower left", ncol=3, fontsize=8)

for i, seed in enumerate(CONFIG["seeds"]):
    axes[1].plot(ticks, pop_matrix[i], alpha=0.55)
axes[1].plot(ticks, pop_matrix.mean(axis=0), color="black", linewidth=2.0)
axes[1].set_ylabel("Effectif Lapin")
axes[1].set_xlabel("Tick")
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Différenciation spatiale (Fst) — quadrants NW / NE / SW / SE

On utilise un seed dédié pour la partie Fst. À chaque checkpoint, on classe les lapins vivants selon leur position dans l'un des 4 quadrants de la carte, puis on calcule Fst entre toutes les paires de quadrants. La courbe Fst(t) doit *croître* dans le temps si la dispersion est insuffisante pour homogénéiser les fréquences alléliques entre quadrants.

In [ ]:
def quadrant_groups(sim, species_name: str, world_size: int) -> dict[str, list]:
    mid = world_size / 2.0
    groups: dict[str, list] = {"NW": [], "NE": [], "SW": [], "SE": []}
    for ind in sim.individuals:
        if ind.species.name != species_name:
            continue
        key = ("N" if ind.y < mid else "S") + ("W" if ind.x < mid else "E")
        groups[key].append(ind.genome)
    return groups

fst_series: dict[str, list[float]] = defaultdict(list)
fst_ticks: list[int] = []

cfg = SimConfig(seed=CONFIG["fst_seed"], grid_size=CONFIG["grid_size"],
                terrain_preset=CONFIG["terrain_preset"], out_path=None)
sim = Simulation(cfg)
for name, opts in CONFIG["species"].items():
    sim.add_species(load_species_params(name), count=opts["count"])

def record_fst():
    fst_ticks.append(sim.tick)
    groups = quadrant_groups(sim, "Lapin", CONFIG["grid_size"])
    keys = [k for k, v in groups.items() if len(v) >= 5]  # quadrants trop petits ignorés
    pairs = [(k1, k2) for i, k1 in enumerate(keys) for k2 in keys[i + 1:]]
    for k1, k2 in pairs:
        fst_series[f"{k1}-{k2}"].append(fst(groups[k1], groups[k2]))
    # combler avec NaN les paires manquantes pour ce tick
    for k in list(fst_series):
        if len(fst_series[k]) < len(fst_ticks):
            fst_series[k].append(float("nan"))

record_fst()
for k in range(1, len(checkpoints)):
    sim.run(checkpoints[k] - checkpoints[k - 1])
    record_fst()
print("Population finale Lapin :", sim.populations.get("Lapin", 0))
print("Paires de quadrants suivies :", list(fst_series.keys()))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
tx = np.asarray(fst_ticks)
for pair, series in fst_series.items():
    ax.plot(tx, series, label=pair, alpha=0.85)
# Moyenne
matrix = np.array([series for series in fst_series.values()])
mean = np.nanmean(matrix, axis=0)
ax.plot(tx, mean, color="black", linewidth=2.0, label="moyenne")
ax.set_xlabel("Tick")
ax.set_ylabel("Fst")
ax.set_title(f"EcoSim {ECOSIM_VERSION} — Différenciation Fst entre quadrants (seed={CONFIG['fst_seed']})")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", ncol=3, fontsize=8)
fig.tight_layout()
plt.show()
print(f"Fst moyen final : {mean[-1]:.4f}")

## 3. Sauvegarde de la configuration

In [ ]:
CONFIG_OUT = Path("02_drift_fst.config.json")
CONFIG_OUT.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(f"Configuration écrite dans {CONFIG_OUT.resolve()}")

## Notes

- **He → 0 attendu sur longue durée.** Sur une population finie soumise à la dérive seule (sans flux génique ni sélection diversifiante), He converge vers 0 selon `He(t) ≈ He(0) · (1 - 1/(2 Ne))^t`. Avec Ne ~ 100 lapins, attendre une demi-vie de 200 générations — sur 4000 ticks, l'effet est partiel mais visible.
- **Fst dépend de la dispersion.** Le `dispersal_radius` du lapin est nul dans le JSON par défaut (seuls les bébés héritent de la position du parent). Augmenter `litter_size_max` ou élargir la grille accentue la différenciation entre quadrants.
- **Mutation.** Le `mutation_rate` (cf. `lapin.json`) gouverne l'apparition de nouvelles allèles ; sans mutation, He chute encore plus vite. Pour explorer l'équilibre mutation-dérive, modifier ce paramètre dans une copie locale du JSON.